In [1]:
import torch
print(torch.__version__)
print("CUDA:", torch.cuda.is_available())


2.11.0+cu128
CUDA: True


## PyTorch hazır mı baktık

---
Tensor + Autograd
hücre 2
⤵



In [2]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3*x + 1
y.backward()

print("y =", y.item())
print("dy/dx =", x.grad)

y = 11.0
dy/dx = tensor(7.)


requires_grad =true >>> Türev Takibi

backward() >>>
backprop otomatik

>>>>>>
PyTorch'ta autograd ile türever otomatik hesaplanır.

---

MODEL + TRAINING LOOP kısmına geçeceğiz

In [3]:
import torch.nn as nn
import torch.optim as optim

#Sahte Veri
x = torch.randn(100, 1)
y = 3*x + 2

#Model
model = nn.Linear(1,1)

criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr = 0.1)

for epoch in range(50):
  optimizer.zero_grad()
  y_pred = model(x)
  loss = criterion(y_pred, y)
  loss.backward()
  optimizer.step()

print("Final Loss:", loss.item())

Final Loss: 1.2952571459834417e-08




1.   Tahmin
2.   Hata


1.   Geri Yayılım
2.   Güncelleme





In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(device)
#Modeli GPU üzerinde eğittk

cuda


In [5]:
#CNN+Görüntü
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.ToTensor()

train_data = datasets.MNIST(
    root=".",
    train=True,
    download=transform
)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)


100%|██████████| 9.91M/9.91M [00:00<00:00, 18.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 516kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.69MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.2MB/s]


In [6]:
train_data = datasets.MNIST(
    root=".",
    train=True,
    download=True,
    transform=transform   # 👈 ŞART
)


In [7]:
train_loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True
)


In [8]:
#CNN Modeli
class CNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv = nn.Conv2d(1,16,3)
    self.pool = nn.MaxPool2d(2,2)
    self.fc = nn.Linear(16*13*13, 10)

  def forward(self, x):
      x = self.pool(torch.relu(self.conv(x)))
      x = x.view(x.size(0), -1)
      return self.fc(x)

In [9]:
del model
model = CNN().to(device)

In [10]:
print(model)

CNN(
  (conv): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=2704, out_features=10, bias=True)
)


In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [12]:
model.train()

for i, (images, labels) in enumerate(train_loader):
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    print("Batch", i+1, "Loss:", loss.item())

    if i == 1:
        break


Batch 1 Loss: 2.2921407222747803
Batch 2 Loss: 2.4241247177124023
